# 対象物アプローチの下降距離分析

pi05 Libero モデル（v44）の対象物直近での推論が、対象物との距離を正しく捉えているかを検証する。

処理の流れは次のとおり。

1. pick&place 系タスクを固定シードで実行し、推論ごとのシミュレータ状態とアクションチャンクを記録する
2. 推論時点の高さの差と、チャンクの下降距離を比較する
3. 異常な下降距離を出したロールアウトを抽出する
4. 抽出した推論を再現し、同じ入力で推論を繰り返して、サンプリングのばらつきと切り分ける

背景と仮説は `doc/theme.md` を参照。

## 1. セットアップ

VM の起動と実行の手順は `doc/colab_vm.md` を参照する。実行前に、次のファイルを VM にアップロードしておく。

| VM 上のパス | 内容 |
|---|---|
| `/content/rclone.conf` | Drive の認証情報。実行の最後に削除する |
| `/content/rollout.py` | `scripts/rollout.py` |

`/content/hf_token` に Hugging Face トークンをアップロードしておく。tokenizer の `google/paligemma-3b-pt-224` は利用申請が必要である。

Drive には `drive.mount` ではなく rclone でアクセスする。

In [ ]:
# VM の cmake は pip の隔離ビルドで動かないため、システムの cmake を先に使う
!apt-get install -y -q cmake > /dev/null
!PATH=/usr/bin:$PATH pip install -q "lerobot[pi,libero]==0.6.1"
!curl -fsSL https://rclone.org/install.sh | bash > /dev/null

In [ ]:
import os
import pathlib
import subprocess
import sys

from huggingface_hub import login

login(token=pathlib.Path("/content/hf_token").read_text().strip())

# ヘッドレス環境で MuJoCo を GPU 描画する
os.environ["MUJOCO_GL"] = "egl"

# Colab の VM には NVIDIA の EGL 定義がない場合があり、CPU 描画に落ちて遅くなるため補う
icd = pathlib.Path("/usr/share/glvnd/egl_vendor.d/10_nvidia.json")
if not icd.exists():
    icd.parent.mkdir(parents=True, exist_ok=True)
    icd.write_text('{"file_format_version":"1.0.0","ICD":{"library_path":"libEGL_nvidia.so.0"}}')

# 実際に描画している実装を確認する。ソフトウェア描画に落ちてもエラーにならず、遅くなるだけのため
# MUJOCO_GL は import 時に確定するので、別プロセスで確認する
probe = """
import mujoco
from OpenGL import GL
context = mujoco.GLContext(256, 256)
context.make_current()
print(GL.glGetString(GL.GL_RENDERER).decode())
"""
renderer = subprocess.run(
    [sys.executable, "-c", probe], capture_output=True, text=True, check=True,
    env=os.environ | {"PYOPENGL_PLATFORM": "egl"},
).stdout.strip()
print("GL_RENDERER:", renderer)
assert "llvmpipe" not in renderer.lower(), "GPU で描画できていない"

In [ ]:
import importlib.util

import yaml

# libero は設定ファイルがないと import 時に input() で止まるため、先に作っておく
libero_root = pathlib.Path(importlib.util.find_spec("libero.libero").origin).parent
config_file = pathlib.Path.home() / ".libero/config.yaml"
config_file.parent.mkdir(exist_ok=True)
config_file.write_text(yaml.dump({
    "benchmark_root": str(libero_root),
    "bddl_files": str(libero_root / "bddl_files"),
    "init_states": str(libero_root / "init_files"),
    "datasets": str(libero_root.parent / "datasets"),
    "assets": str(libero_root / "assets"),
}))

## 2. 共通処理

設定、モデル、環境、ロールアウトの処理は `scripts/rollout.py` にある。各ワーカーと第 6 章の再現は、同じコードを使う。実行前に、VM の `/content/rollout.py` へアップロードしておく。

同じロールアウトが並列の割り当てや実行順に関係なく同じ結果になるよう、`run_episode` の先頭で種を蒔き直している。推論の初期ノイズは `(シード, スイート, タスク, エピソード, ステップ, 試行番号)` から決める。

## 3. データ収集

`NUM_WORKERS` 個のプロセスで分担して実行する。ログは `results/worker*.log` に出る。

ファイル数が多いため、結果は 1 個のアーカイブにまとめて定期的に Drive へ送る。保存済みのエピソードは飛ばすので、VM が回収されても新しい VM で実行し直せば続きから収集できる。

In [ ]:
import subprocess
import sys
import time

sys.path.insert(0, "/content")
from huggingface_hub import snapshot_download
from transformers import AutoTokenizer

from rollout import *

SYNC_INTERVAL_S = 600


def sync_to_drive():
    """結果をアーカイブにまとめて Drive に送る。"""
    subprocess.run(["tar", "czf", str(ARCHIVE), "-C", str(LOCAL_DIR.parent), LOCAL_DIR.name], check=True)
    subprocess.run(["rclone", "copyto", str(ARCHIVE), REMOTE_ARCHIVE, "--config", RCLONE_CONF], check=True)


# Drive に前回の結果があれば取得する。初回は存在しないので失敗してよい
subprocess.run(["rclone", "copyto", REMOTE_ARCHIVE, str(ARCHIVE), "--config", RCLONE_CONF], check=False)
if ARCHIVE.exists():
    subprocess.run(["tar", "xzf", str(ARCHIVE), "-C", str(LOCAL_DIR.parent)], check=True)

# ワーカーが同時にダウンロードしないよう、モデルとアセットを先に取得する
snapshot_download(MODEL_ID)
AutoTokenizer.from_pretrained("google/paligemma-3b-pt-224")
make_env(SUITES[0], 0)[0].close()

logs = [open(LOCAL_DIR / f"worker{i}.log", "w") for i in range(NUM_WORKERS)]
workers = [
    subprocess.Popen([sys.executable, "/content/rollout.py", str(i)], stdout=log, stderr=subprocess.STDOUT)
    for i, log in enumerate(logs)
]

while any(worker.poll() is None for worker in workers):
    time.sleep(SYNC_INTERVAL_S)
    sync_to_drive()
    print(f"{time.strftime('%H:%M')} 完了: {len(list(EPISODE_DIR.glob('*.npz')))} / {len(all_rollouts())}", flush=True)

sync_to_drive()
print("ワーカーの終了コード:", [worker.returncode for worker in workers])
assert all(worker.returncode == 0 for worker in workers), "ログ results/worker*.log を確認する"

## 4. 下降距離の分析

グリッパー指令は 1 ステップで閉へ切り替わるが、手先はその後も数ステップ下降を続ける。そのため、閉指令の開始後に z 指令が下降でなくなる点を**アプローチ完了**とする。この定義は、実行した軌跡とチャンクの両方に同じように適用する。

分析には、実行軌跡のアプローチ完了より前の推論を使う。閉指令の開始後の推論も含むが、配置先への下降は混ざらない。

タスクの成否は問わない。ただし、対象物に降りないまま終わったロールアウトは比較できないので除く。アプローチ完了時の手先が、対象物から水平に `REACH_RADIUS_M` 以内、高さの差が `REACH_HEIGHT_M` 以内のものだけを使う。届かないケースも検出できるよう、高さの条件は緩くしている。

下降量はチャンク全量からアプローチ完了までの範囲で計算するので、実際に実行したステップ数（推論間隔）には依存しない。

### アクションからメートルへの換算

`GAIN_WINDOW_STEPS` ごとの固定窓で、実際に動いた高さと z アクションの合計を最小二乗法で当てはめ、換算係数を実測する。

### 必要な下降距離

対象物の形状によって把持する高さが違うため、実績から求める。

- `把持オフセット` = 対象物に降りたロールアウトの、アプローチ完了時の `手先の高さ - 対象物の高さ` の、タスクごとの中央値
- `必要な下降距離` = `手先の高さ - 対象物の高さ - 把持オフセット`

### チャンクの下降距離と誤差

- `計画下降距離` = チャンクの先頭からアプローチ完了までの、z アクションの合計を換算した値
- `誤差` = `計画下降距離 - 必要な下降距離`。正なら突っ込みすぎ、負なら届かない

アプローチ完了までを計画していないチャンクは下降距離を比較できないため、分析から除く。

In [ ]:
import pandas as pd


def approach_end(actions):
    """閉指令の開始後に z 指令が下降でなくなるインデックスを返す。到達しなければ None を返す。"""
    closing = np.flatnonzero(actions[:, 6] > 0)
    if len(closing) == 0:
        return None
    close = closing[0]
    not_descending = np.flatnonzero(actions[close:, 2] >= 0)
    return int(close + not_descending[0]) if len(not_descending) else None


def load_episodes():
    episodes = []
    for path in sorted(EPISODE_DIR.glob("*.npz")):
        with np.load(path) as data:
            episode = {key: data[key] for key in data.files}
        for key in ["suite", "target"]:
            episode[key] = str(episode[key])
        for key in ["task_id", "episode"]:
            episode[key] = int(episode[key])
        episode["success"] = bool(episode["success"])
        episode["end_t"] = approach_end(episode["step_action"])
        episodes.append(episode)
    return episodes


def reached_target(ep):
    """アプローチ完了時に、手先が対象物の近くにいれば True を返す。"""
    if ep["end_t"] is None:
        return False
    offset = ep["step_eef"][ep["end_t"]] - ep["step_obj"][ep["end_t"]]
    return np.linalg.norm(offset[:2]) <= REACH_RADIUS_M and offset[2] <= REACH_HEIGHT_M


def estimate_gain(episodes):
    """z アクションの合計に対する、実際の高さ変化の比を返す。"""
    commanded, moved = [], []
    for ep in episodes:
        end = ep["end_t"] if ep["end_t"] is not None else len(ep["step_action"])
        for t in range(0, end - GAIN_WINDOW_STEPS, GAIN_WINDOW_STEPS):
            commanded.append(ep["step_action"][t:t + GAIN_WINDOW_STEPS, 2].sum())
            moved.append(ep["step_eef"][t + GAIN_WINDOW_STEPS, 2] - ep["step_eef"][t, 2])
    commanded, moved = np.array(commanded), np.array(moved)
    gain = commanded @ moved / (commanded @ commanded)
    r2 = np.corrcoef(commanded, moved)[0, 1] ** 2
    return gain, r2


def grasp_offsets(episodes):
    """タスクごとの把持オフセットを返す。"""
    rows = [
        {"suite": ep["suite"], "task_id": ep["task_id"],
         "offset": ep["step_eef"][ep["end_t"], 2] - ep["step_obj"][ep["end_t"], 2]}
        for ep in episodes if reached_target(ep)
    ]
    return pd.DataFrame(rows).groupby(["suite", "task_id"])["offset"].median().to_dict()


def planned_descent(chunk, gain):
    """チャンクの先頭からアプローチ完了までの下降距離を返す。完了まで計画していなければ None を返す。"""
    end = approach_end(chunk)
    if end is None:
        return None
    return -gain * np.clip(chunk[:end, 2], -1.0, 1.0).sum()


def build_table(episodes, gain, offsets):
    """アプローチ完了より前の推論を、1 推論 1 行の表にする。"""
    rows = []
    for ep in episodes:
        if not reached_target(ep):
            continue
        offset = offsets[(ep["suite"], ep["task_id"])]
        # アプローチ完了時に、手先が基準の高さからどれだけずれていたか。正なら高い位置で止まった
        end_gap = ep["step_eef"][ep["end_t"], 2] - ep["step_obj"][ep["end_t"], 2] - offset
        for index, t in enumerate(ep["infer_t"]):
            planned = planned_descent(ep["chunks"][index], gain)
            if t >= ep["end_t"] or planned is None:
                continue
            required = ep["step_eef"][t, 2] - ep["step_obj"][t, 2] - offset
            rows.append({
                "suite": ep["suite"], "task_id": ep["task_id"], "episode": ep["episode"], "t": int(t),
                "success": ep["success"], "required": required, "planned": planned,
                "error": planned - required, "end_gap": end_gap,
            })
    return pd.DataFrame(rows)


episodes = load_episodes()
gain, r2 = estimate_gain(episodes)
offsets = grasp_offsets(episodes)
table = build_table(episodes, gain, offsets)

print(f"エピソード数: {len(episodes)}, 成功率: {np.mean([ep['success'] for ep in episodes]):.1%}")
print(f"換算係数: {gain:.4f} m/アクション (R2={r2:.3f})")
print(f"分析対象の推論数: {len(table)}")
print(f"対象物に降りなかったロールアウト: {sum(not reached_target(ep) for ep in episodes)}")
print(f"対象物に降りた例がないタスク: {sorted({(ep['suite'], ep['task_id']) for ep in episodes} - set(offsets))}")

### 高さの差ごとの誤差

仮説が正しければ、必要な下降距離が小さい（対象物に近い）区間ほど、誤差の広がりが大きくなるはずである。

In [ ]:
import matplotlib.pyplot as plt

HEIGHT_BINS_M = [-np.inf, 0.01, 0.02, 0.04, 0.06, 0.08, 0.12, np.inf]

table["height_bin"] = pd.cut(table["required"], HEIGHT_BINS_M)
summary = table.groupby("height_bin", observed=True)["error"].agg(
    count="count", mean="mean", std="std",
    p5=lambda x: x.quantile(0.05), p95=lambda x: x.quantile(0.95),
)
summary["abnormal_rate"] = table.groupby("height_bin", observed=True)["error"].apply(
    lambda x: (x.abs() > ERROR_THRESHOLD_M).mean()
)
print(summary.round(4))
summary.to_csv(LOCAL_DIR / "error_by_height.csv")

fig, ax = plt.subplots(figsize=(7, 4))
for success, marker, label in [(True, "o", "success"), (False, "x", "failure")]:
    subset = table[table["success"] == success]
    ax.scatter(subset["required"] * 100, subset["error"] * 100, s=10, marker=marker, alpha=0.5, label=label)
ax.axhline(0, color="gray", linewidth=1)
ax.set_xlabel("Required descent [cm]")
ax.set_ylabel("Planned - required [cm]  (+: overshoot, -: undershoot)")
ax.legend()
fig.tight_layout()
fig.savefig(LOCAL_DIR / "error_by_height.png", dpi=150)
plt.show()

## 5. 問題ロールアウトの抽出

誤差の絶対値が `ERROR_THRESHOLD_M` を超えた推論を異常とみなし、エピソードごとに誤差が最大の推論を 1 つ選ぶ。

`end_gap` は、実際のアプローチ完了時の高さの基準からのずれである。正なら届かないまま止まった、負なら基準より深く入ったことを示す。

In [ ]:
abnormal = table[table["error"].abs() > ERROR_THRESHOLD_M]
worst_index = abnormal["error"].abs().groupby([abnormal["suite"], abnormal["task_id"], abnormal["episode"]]).idxmax()
problems = table.loc[worst_index].drop(columns="height_bin")
problems = problems.sort_values("error", key=np.abs, ascending=False).reset_index(drop=True)

problems.to_csv(LOCAL_DIR / "problem_rollouts.csv", index=False)
print(f"異常な推論を含むエピソード: {len(problems)} / {len(episodes)}")
problems.head(20).round(4)

## 6. 問題ロールアウトの再現と繰り返し推論

`problems` の上位について、固定シードで該当ステップまで再現する。再現した状態が記録と一致することを確認してから、シミュレータを進めずに乱数だけを変えて推論を繰り返す。

`repeat_mean` の誤差も閾値を超えていれば、たまたまの外れ値ではなく、その状態で一貫して距離を誤っていると判断できる。

In [ ]:
def repeat_inference(row):
    """問題の推論を再現し、REPEATS 回の推論の下降距離を返す。"""
    suite_name, task_id, episode, t = row.suite, row.task_id, row.episode, row.t
    env, init_states, prompt, target = make_env(suite_name, task_id)
    obs = run_episode(env, init_states, prompt, target, suite_name, task_id, episode, stop_at=t)

    # 再現した状態を記録と照合する
    with np.load(episode_path(suite_name, task_id, episode)) as data:
        recorded = data["infer_state"][list(data["infer_t"]).index(t)]
    reproduced = np.allclose(env.get_sim_state(), recorded, atol=1e-6)

    descents = []
    for repeat in range(1, REPEATS + 1):
        chunk = infer_chunk(obs, prompt, rollout_seed(suite_name, task_id, episode, t, repeat))
        descents.append(planned_descent(chunk, gain))
    env.close()
    # アプローチ完了まで計画していない試行は比較できないため除く
    return reproduced, np.array([d for d in descents if d is not None])


load_policy()

rows = []
for row in problems.head(TOP_N).itertuples():
    reproduced, descents = repeat_inference(row)
    if not reproduced:
        print(f"状態が一致しないためスキップ: {row.suite} task{row.task_id} ep{row.episode} t={row.t}")
        continue
    repeat_error = descents - row.required
    rows.append({
        "suite": row.suite, "task_id": row.task_id, "episode": row.episode, "t": row.t,
        "required": row.required, "original_error": row.error, "valid_repeats": len(descents),
        "repeat_mean": repeat_error.mean(), "repeat_std": repeat_error.std(),
        "consistent": abs(repeat_error.mean()) > ERROR_THRESHOLD_M,
    })

repeats = pd.DataFrame(rows)
repeats.to_csv(LOCAL_DIR / "repeat_inference.csv", index=False)
sync_to_drive()
repeats.round(4)

## 7. 後始末

Drive の認証情報と Hugging Face トークンを VM から削除する。VM の停止は手元から `colab stop` で行う。

In [ ]:
!shred -u /content/rclone.conf /content/hf_token